In [ ]:
import pandas as pd
import subprocess
import numpy as np

In [ ]:
data = 'Data/Fe56.csv'
data_deleted = 'Data/Fe56_Deleted.csv'
data_fit = 'Data/Fe56_Fit.csv'
elem = 'Fe56'
mass_nucleon = 0.938273
Veff = 0.0089
ex_cut_lower = 0
ex_cut_upper = 1000
multiplier = 26 / 6
QEMEC_shift = 0.0
Use_deleted = True

In [ ]:
if Use_deleted:
    df_valid = pd.read_csv(data)
    df_invalid = pd.read_csv(data_deleted)
    df = pd.concat([df_valid, df_invalid], ignore_index=True)
else:
    df = pd.read_csv(data)

In [ ]:
if QEMEC_shift == 0.0:
    with open(data_fit, "w") as response_file:
        response_file.write('Z,A,E0,ThetaDeg,nu,cross,error,dataSet,sigtot,sigqe,sigie,sigmec,nuccstot,signonuc,ratio\n')
    for index, row in df.iterrows():
        Z = row['Z']
        A = row['A']
        E0 = row['E0']
        ThetaDeg = row['ThetaDeg']
        nu = row['nu']
        cross = row['cross']
        error = row['error']
        dataSet = row['dataSet']
        if nu <= QEMEC_shift:
            continue
        with open("input1.txt", "w") as input1:
            input1.write(f"{E0} {ThetaDeg}\n")
        with open("input2.txt", "w") as input2:
            input2.write(f"{nu}\n")
        with open("output1.txt", "w") as output_file:
            subprocess.run(["qemodplot/qemodplot.exe"], stdout=output_file)
        with open("output1.txt", "r") as output_file:
            line = output_file.readline().strip()
            values = line.split()
        if not values:
            continue
        for i in range(6, 12):
            values[i] = float(values[i]) * 1000 * 12 * multiplier
        Ex = float(values[3])
        if Ex < ex_cut_lower or Ex > ex_cut_upper:
            continue
        sigtot = values[6]
        sigqe = values[7]
        sigie = values[8]
        sigmec = values[9]
        nuccstot = values[10]
        signonuc = values[11]
        if sigtot == 0:
            ratio = 0
        else:
            ratio = float(cross) / sigtot
        with open(data_fit, "a") as response_file:
            response_file.write(f"{Z},{A},{E0},{ThetaDeg},{nu},{cross},{error},{dataSet},{sigtot},{sigqe},{sigie},{sigmec},{nuccstot},{signonuc},{ratio}\n")
    df_fit = pd.read_csv(data_fit)
    data = df_fit['cross']
    fit = df_fit['sigtot']
    error = df_fit['error']
    chi_square_components = ((data - fit) ** 2) / (error ** 2)
    total_chi_square = np.sum(chi_square_components)
    print(f"Total Chi-Square: {total_chi_square}")
    
else:
    with open(data_fit, "w") as response_file:
        response_file.write('Z,A,E0,ThetaDeg,nu,cross,error,dataSet,sigtot,sigqe,sigie,sigmec,nuccstot,signonuc,sigtot_shifted,qe_shifted,ratio\n')
    for index, row in df.iterrows():
        Z = row['Z']
        A = row['A']
        E0 = row['E0']
        ThetaDeg = row['ThetaDeg']
        nu = row['nu']
        cross = row['cross']
        error = row['error']
        dataSet = row['dataSet']
        if nu <= QEMEC_shift:
            continue
        with open("input1.txt", "w") as input1:
            input1.write(f"{E0} {ThetaDeg}\n")
        with open("input2.txt", "w") as input2:
            input2.write(f"{nu}\n")
        with open("output1.txt", "w") as output_file:
            subprocess.run(["qemodplot/qemodplot.exe"], stdout=output_file)
        with open("input2.txt", "w") as input2:
            input2.write(f"{nu - QEMEC_shift}\n")
        with open("output2.txt", "w") as output_file:
            subprocess.run(["qemodplot/qemodplot.exe"], stdout=output_file)
        with open("output1.txt", "r") as output_file:
            line = output_file.readline().strip()
            values = line.split()
        with open("output2.txt", "r") as output_file:
            line = output_file.readline().strip()
            values_shifted = line.split()
        if not values:
            continue
        for i in range(6, 12):
            values[i] = float(values[i]) * 1000 * 12 * multiplier
            values_shifted[i] = float(values_shifted[i]) * 1000 * 12 * multiplier
        Ex = float(values[3])
        if Ex < ex_cut_lower or Ex > ex_cut_upper:
            continue
        sigtot = values[6]
        sigqe = values[7]
        sigie = values[8]
        sigmec = values[9]
        nuccstot = values[10]
        signonuc = values[11]
        sigtot_shifted = values[6] - values[7] + values_shifted[7] - values[9] + values_shifted[9]
        qemec_shifted = values_shifted[7] + values_shifted[9]
        if sigtot_shifted == 0:
            ratio = 0
        else:
            ratio = float(cross) / sigtot_shifted
        with open(data_fit, "a") as response_file:
            response_file.write(f"{Z},{A},{E0},{ThetaDeg},{nu},{cross},{error},{dataSet},{sigtot},{sigqe},{sigie},{sigmec},{nuccstot},{signonuc},{sigtot_shifted},{qemec_shifted},{ratio}\n")
    df_fit = pd.read_csv(data_fit)
    data = df_fit['cross']
    fit = df_fit['sigtot_shifted']
    error = df_fit['error']
    chi_square_components = ((data - fit) ** 2) / (error ** 2)
    total_chi_square = np.sum(chi_square_components)
    print(f"Total Chi-Square: {total_chi_square}")